# Simple LangChain PDF Question-Answering Bot

A small project to show hands-on LangChain experience: load a PDF, split it into
chunks, embed and store it in FAISS, then ask questions and get answers grounded
only in the PDF's content.

**Document used:** A short PDF about the Solar System (created for this demo).

**LangChain pieces used:**
- `PyPDFLoader` - load the PDF
- `RecursiveCharacterTextSplitter` - split into chunks
- `HuggingFaceEmbeddings` - turn chunks into vectors
- `FAISS` - store and search vectors
- `ChatOpenAI` - generate answers (via GitHub Models, OpenAI-compatible endpoint)
- LCEL chain (`retriever | prompt | llm | parser`) - ties it all together


## 1. Install & Import Libraries

In [1]:
# Run this once if any of these are missing:
# pip install langchain langchain-community langchain-openai faiss-cpu pypdf sentence-transformers python-dotenv
!pip install langchain langchain-community langchain-openai faiss-cpu pypdf sentence-transformers python-dotenv

   ---------------------------------------- 0.0/558.3 kB ? eta -:--:--
   ------------------------------------- -- 524.3/558.3 kB 3.1 MB/s eta 0:00:01
   ---------------------------------------- 558.3/558.3 kB 2.1 MB/s  0:00:00
   ---------------------------------------- 0.0/653.4 kB ? eta -:--:--
   ---------------------------------------- 653.4/653.4 kB 3.2 MB/s  0:00:00
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ----------------- ---------------------- 1.0/2.4 MB 9.1 MB/s eta 0:00:01
   ----------------------------------- ---- 2.1/2.4 MB 6.7 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 6.1 MB/s  0:00:00
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 6.3 MB/s  0:00:00
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------------------------- -------------- 1.0/1.6 MB 5.9 MB/s eta 0:00:01
   ----------------------------------------

In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import os
from dotenv import load_dotenv

load_dotenv()
print("Libraries loaded.")


C:\Users\shrey\AppData\Local\Temp\ipykernel_19496\1332613850.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Libraries loaded.


## 2. Set Up API Key

This uses the same GitHub Models endpoint you already used in your RAG Teaching
Assistant project (OpenAI-compatible, free tier, `gpt-4.1-mini`).

Create a `.env` file in this folder with:
```
GITHUB_TOKEN=your_github_models_token_here
```


In [3]:
GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

llm = ChatOpenAI(
    model="gpt-4.1-mini",
    api_key=GITHUB_TOKEN,
    base_url="https://models.inference.ai.azure.com",
    temperature=0,
)

print("LLM configured.")


LLM configured.


## 3. Load the PDF

We use a short sample PDF about the Solar System.


In [4]:
loader = PyPDFLoader("solar_system.pdf")
documents = loader.load()

print(f"Pages loaded: {len(documents)}")
print(documents[0].page_content[:300])


Pages loaded: 2
The Solar System - A Short Guide
Introduction
The Solar System consists of the Sun and everything that orbits around it, including eight planets, their
moons, dwarf planets, asteroids, comets, and meteoroids. It formed about 4.6 billion years ago from the
gravitational collapse of a giant interstell


## 4. Split the Document into Chunks

Large documents need to be split into smaller pieces so the retriever can find
the most relevant section for a given question, instead of dumping the whole
document into every prompt.


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)

chunks = splitter.split_documents(documents)
print(f"Total chunks: {len(chunks)}")
print(chunks[0].page_content)


Total chunks: 8
The Solar System - A Short Guide
Introduction
The Solar System consists of the Sun and everything that orbits around it, including eight planets, their
moons, dwarf planets, asteroids, comets, and meteoroids. It formed about 4.6 billion years ago from the
gravitational collapse of a giant interstellar molecular cloud.
The Sun
The Sun is a nearly perfect sphere of hot plasma at the center of the Solar System. It contains 99.86


## 5. Create Embeddings and Store in FAISS

Each chunk gets converted into a vector (a list of numbers representing its
meaning), then stored in FAISS for fast similarity search. We use the same
multilingual MiniLM model from your RAG Teaching Assistant project.


In [6]:
embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")

vectorstore = FAISS.from_documents(chunks, embeddings)
print("Vector store created.")


C:\Users\shrey\AppData\Local\Temp\ipykernel_19496\2509398511.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="paraphrase-multilingual-MiniLM-L12-v2")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vector store created.


In [7]:
# Save it locally so we don't have to rebuild it every time
vectorstore.save_local("solar_system_index")
print("Vector store saved to disk.")


Vector store saved to disk.


## 6. Create the Retriever

The retriever's job: given a question, find the most relevant chunks from the
vector store.


In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# Quick test - see what chunks get retrieved for a sample question
results = retriever.invoke("What is Jupiter known for?")
for i, doc in enumerate(results):
    print(f"--- Chunk {i+1} ---")
    print(doc.page_content[:200])
    print()


--- Chunk 1 ---
Jupiter
Jupiter is the largest planet in the Solar System, more massive than all the other planets combined. It is
a gas giant made mostly of hydrogen and helium, and is famous for its Great Red Spot,

--- Chunk 2 ---
The Solar System - A Short Guide
Introduction
The Solar System consists of the Sun and everything that orbits around it, including eight planets, their
moons, dwarf planets, asteroids, comets, and met

--- Chunk 3 ---
Neptune
Neptune is the farthest known planet from the Sun and is also an ice giant. It has the strongest winds in
the Solar System, reaching speeds of over 2,000 kilometers per hour. Neptune was the f



## 7. Build the Prompt Template

This tells the LLM exactly how to use the retrieved chunks to answer the
question - and to only use the given context, not outside knowledge.


In [9]:
prompt = ChatPromptTemplate.from_template(
    """Answer the question using ONLY the context below.
If the answer is not in the context, say "I don't have that information in the document."

Context:
{context}

Question: {question}

Answer:"""
)


## 8. Build the Full Chain (LCEL)

This is the LangChain way of writing: retrieve context -> build prompt -> call LLM
-> parse output, as one connected pipeline.


In [10]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain ready.")


RAG chain ready.


## 9. Ask Questions

Now let's test the full pipeline with a few questions.


In [11]:
question = "Which planet has the strongest winds?"
answer = rag_chain.invoke(question)

print(f"Q: {question}")
print(f"A: {answer}")


Q: Which planet has the strongest winds?
A: Neptune has the strongest winds in the Solar System, reaching speeds of over 2,000 kilometers per hour.


In [12]:
question = "Why is Venus the hottest planet even though Mercury is closer to the Sun?"
answer = rag_chain.invoke(question)

print(f"Q: {question}")
print(f"A: {answer}")


Q: Why is Venus the hottest planet even though Mercury is closer to the Sun?
A: Venus is the hottest planet because it has a thick, toxic atmosphere made mostly of carbon dioxide, causing a runaway greenhouse effect that traps heat. Mercury, although closer to the Sun, has no atmosphere to retain heat, so its temperatures swing from extremely hot during the day to extremely cold at night.


In [13]:
question = "What is the capital of France?"  # not in the document - tests grounding
answer = rag_chain.invoke(question)

print(f"Q: {question}")
print(f"A: {answer}")


Q: What is the capital of France?
A: I don't have that information in the document.


## 10. Try It Yourself

Type your own question about the Solar System below.


In [14]:
my_question = "How many known moons does Jupiter have?"   # <-- change this line

answer = rag_chain.invoke(my_question)
print(f"Q: {my_question}")
print(f"A: {answer}")


Q: How many known moons does Jupiter have?
A: Jupiter has at least 95 known moons.


## Summary

- Loaded a PDF using `PyPDFLoader`
- Split it into chunks using `RecursiveCharacterTextSplitter`
- Converted chunks into vectors using `HuggingFaceEmbeddings` and stored them in `FAISS`
- Built a retriever to fetch relevant chunks for any question
- Connected everything into one LCEL chain: `retriever -> prompt -> llm -> parser`
- Tested with real questions, including one outside the document to confirm the
  model stays grounded and doesn't hallucinate an answer

This demonstrates the same RAG concept as the hand-coded Teaching Assistant
project, but built using LangChain's framework components instead of writing
each piece (embedding, vector store, retrieval, prompting) manually.
